# 6. Compute·State·Environment 분리와 Watchdog

근거: [설계 글 §5–6](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 무엇이 재시작 후 남아야 하는가
compute는 교체될 수 있고 상태는 남아야 합니다. 실행환경은 별도 서비스로 접근합니다. 아래에서는 SQLite 파일(상태), Python 함수(compute), 임시 폴더 provider(환경)를 분리합니다. 원격 daemon이나 실제 클라우드를 띄우는 예제는 아닙니다.

저장한 사건 ID로 중복을 거부하고 버전 번호로 오래된 결과를 거부합니다. 데이터베이스 쓰기의 멱등성만으로 외부 부작용의 exactly-once가 보장되지는 않습니다.

In [1]:
import sqlite3, tempfile
from pathlib import Path
sandbox=tempfile.TemporaryDirectory()
root=Path(sandbox.name); db=root/'state.db'
def connect():
    c=sqlite3.connect(db)
    c.execute('CREATE TABLE IF NOT EXISTS events (id TEXT PRIMARY KEY, text TEXT)')
    c.execute('CREATE TABLE IF NOT EXISTS jobs (id TEXT PRIMARY KEY, revision INTEGER, status TEXT, progress INTEGER, heartbeat INTEGER, retries INTEGER)')
    return c
with connect() as c:
    c.execute('INSERT OR IGNORE INTO events VALUES (?,?)',('e1','조사 요청'))
    c.execute('INSERT OR IGNORE INTO events VALUES (?,?)',('e1','중복'))
    c.execute('INSERT INTO jobs VALUES (?,?,?,?,?,?)',('j1',1,'running',0,100,0))
# compute를 다시 만들었다고 가정하고 새로운 연결로 복원
with connect() as fresh:
    assert fresh.execute('SELECT count(*) FROM events').fetchone()[0]==1
    assert fresh.execute('SELECT text FROM events').fetchone()[0]=='조사 요청'
    # 늦은 revision=0 작업 결과는 반영하지 않음
    changed=fresh.execute("UPDATE jobs SET status='completed' WHERE id=? AND revision=?",('j1',0)).rowcount
    assert changed==0
print('새 DB 연결에서 상태 복원, 중복·오래된 결과 거부')

새 DB 연결에서 상태 복원, 중복·오래된 결과 거부


In [2]:
from typing import Protocol
class Environment(Protocol):
    def read_note(self) -> str: ...
class FolderEnvironment:
    def __init__(self,folder): self.folder=folder
    def read_note(self): return (self.folder/'note.txt').read_text()
def compute_summary(environment: Environment):
    return '관찰: '+environment.read_note()
local=root/'local'; remote_sim=root/'remote-simulation'
local.mkdir(); remote_sim.mkdir()
(local/'note.txt').write_text('로컬 미커밋 변경')
(remote_sim/'note.txt').write_text('다른 실행환경의 문서')
assert compute_summary(FolderEnvironment(local))=='관찰: 로컬 미커밋 변경'
assert compute_summary(FolderEnvironment(remote_sim))=='관찰: 다른 실행환경의 문서'
print('compute 변경 없이 환경 provider 교체; 두 번째 폴더도 실제로는 로컬입니다.')

compute 변경 없이 환경 provider 교체; 두 번째 폴더도 실제로는 로컬입니다.


## 살아 있는 프로세스도 진전이 없을 수 있다
원문의 Watchdog은 주 시스템 밖에서 예약 메시지로 구동되는 독립 agent session이며 의미적 진전을 판단합니다. 여기서는 그 중 **외부 관찰, bounded recovery, 진전과 생존 구분**을 결정적 규칙으로 실습합니다. 이 규칙이 실제 목표 이탈을 이해하는 LLM Watchdog이라는 뜻은 아닙니다. 시계를 주입해 sleep 없이 테스트합니다.

In [3]:
def watchdog(db_path,now):
    with sqlite3.connect(db_path) as c:
        row=c.execute('SELECT id,progress,heartbeat,retries FROM jobs WHERE status="running"').fetchone()
        if not row: return 'idle'
        job,progress,heartbeat,retries=row
        if now-heartbeat>20: cause='liveness'
        elif now-progress>30: cause='no-progress'
        else: return 'healthy'
        if retries>=1:
            c.execute('UPDATE jobs SET status="needs-human" WHERE id=?',(job,))
            return 'escalated:'+cause
        c.execute('UPDATE jobs SET retries=retries+1, revision=revision+1, progress=?, heartbeat=? WHERE id=?',(now,now,job))
        return 'retry:'+cause
assert watchdog(db,100)=='retry:no-progress'
assert watchdog(db,105)=='healthy'
assert watchdog(db,140)=='escalated:liveness'
with connect() as c:
    assert c.execute('SELECT status FROM jobs').fetchone()[0]=='needs-human'
print('진전 없는 작업 감지 → 한 번 회복 시도 → 재실패 시 escalation')
sandbox.cleanup()

진전 없는 작업 감지 → 한 번 회복 시도 → 재실패 시 escalation


## LangGraph와 매핑
MemorySaver는 프로세스 메모리이므로 프로세스 장애 후 내구성을 주장할 수 없습니다. 운영에서는 영속 checkpointer와 사건 inbox/outbox, job store가 필요합니다. BYOM 실행환경은 인증·접근 제어·기기 오프라인 처리·credential 격리가 추가로 필요합니다. 로컬 폴더 provider를 원격 실행이라고 부르지 않습니다.

## 직접 바꿔 보기

heartbeat만 1초마다 갱신하는 무한 반복 작업을 상상해 보세요. progress는 무엇으로 측정해야 할까요?

## 해설과 참고 답안

도구 호출 수만 늘어나는 것은 진전이 아닐 수 있습니다. 통과한 테스트 수, 검증된 산출물, 완료된 목표 등 업무에 맞는 증거를 기록해야 합니다. 의미 판단 실패를 고려해 복구 횟수·시간 예산과 사람에게 넘기는 경로를 둡니다.